In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

from scipy.stats import pearsonr

import torch
import torch.nn as nn

from bayes_opt import BayesianOptimization

In [2]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, lstm_dropout, fc_dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=lstm_dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(fc_dropout)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        output = self.fc(self.dropout(out[:, -1, :]))
        return output

In [3]:
df = pd.read_csv("nn_cluster0_num_of_deliveries.csv")
df.head()

,Order_Date,Number_of_Deliveries
0,2022-03-01,5
1,2022-03-02,1
2,2022-03-03,5
3,2022-03-04,0
4,2022-03-05,4


In [4]:
df["Day"] = pd.to_datetime(df["Order_Date"], format="%Y-%m-%d").dt.dayofweek
df.head()

,Order_Date,Number_of_Deliveries,Day
0,2022-03-01,5,1
1,2022-03-02,1,2
2,2022-03-03,5,3
3,2022-03-04,0,4
4,2022-03-05,4,5


In [5]:
data = df[["Number_of_Deliveries", "Day"]].astype(float).to_numpy()

In [6]:
sequence_length = 3
sequences = []
for i in range(len(data)-sequence_length):
    seq = data[i:i+sequence_length]
    lbl = data[i+sequence_length][0]
    sequences.append((seq, lbl))

In [7]:
sequences

[(array([[5., 1.],
         [1., 2.],
         [5., 3.]]),
  np.float64(0.0)),
 (array([[1., 2.],
         [5., 3.],
         [0., 4.]]),
  np.float64(4.0)),
 (array([[5., 3.],
         [0., 4.],
         [4., 5.]]),
  np.float64(1.0)),
 (array([[0., 4.],
         [4., 5.],
         [1., 6.]]),
  np.float64(5.0)),
 (array([[4., 5.],
         [1., 6.],
         [5., 0.]]),
  np.float64(1.0)),
 (array([[1., 6.],
         [5., 0.],
         [1., 1.]]),
  np.float64(4.0)),
 (array([[5., 0.],
         [1., 1.],
         [4., 2.]]),
  np.float64(0.0)),
 (array([[1., 1.],
         [4., 2.],
         [0., 3.]]),
  np.float64(3.0)),
 (array([[4., 2.],
         [0., 3.],
         [3., 4.]]),
  np.float64(1.0)),
 (array([[0., 3.],
         [3., 4.],
         [1., 5.]]),
  np.float64(5.0)),
 (array([[3., 4.],
         [1., 5.],
         [5., 6.]]),
  np.float64(1.0)),
 (array([[1., 5.],
         [5., 6.],
         [1., 0.]]),
  np.float64(4.0)),
 (array([[5., 6.],
         [1., 0.],
         [4., 

In [ ]:
X = np.array([seq[0] for seq in sequences])
y = np.array([[lbl[1]] for lbl in sequences]) 

In [ ]:
X_train, X_next, y_train, y_next = train_test_split(X, y, test_size=0.3, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_next, y_next, test_size=0.5, shuffle=False)

In [ ]:
X_train.shape

In [ ]:
X_train

In [ ]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train[:, :, 0] = scaler_X.fit_transform(X_train[:, :, 0])

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
model = LSTMModel(2, 40, 2, 1, 0, 0)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03, weight_decay=0)

In [ ]:
train_losses = []
val_losses = []

In [ ]:
for epoch in range(120):
    model.train()
    output = model(X_train)
    loss = criterion(output, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        val_output = model(X_val)
        val_loss = criterion(val_output, y_val)
        val_losses.append(val_loss.item())

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/100], Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}')

In [ ]:
model.eval()
with torch.no_grad():
    test_output = model(X_test)
    test_loss = criterion(test_output, y_test)
    print(f"Test Loss : {test_loss.item()}")

In [ ]:
pearson_corr, _ = pearsonr(test_output, y_test)
mae = mean_absolute_error(y_test, test_output)
mse = mean_squared_error(y_test, test_output)
rmse = np.sqrt(mse)
print(f"MSE Loss: {mse}")
print(f"RMSE Loss: {rmse}")
print(f"MAE Loss: {mae}")
print(f"Pearson's Correlation Coefficient: {pearson_corr.item()}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(y_test.numpy(), label="Actual Values", marker="o", linestyle="--", color="tab:orange", markersize=6)
plt.plot(test_output.numpy(), label="Predicted Values", marker="x", linestyle="--", color="tab:blue", markersize=6)
plt.xlabel("Days", fontsize=12)
plt.ylabel("Number of Deliveries", fontsize=12)
plt.legend(loc="best", fontsize=10, frameon=True, shadow=True)
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()
plt.close()